In [4]:
import sys
import os
import numpy as  np
from itertools import combinations
import networkx as nx
import matplotlib.pyplot as plt


# Get the root directory (QAOA_mtrix_simulation)File ~\OneDrive - Nottingham Trent University\Desktop\Projects\QAOA_simulation_LV\QAOA_mtrix_simulation\main\Base\__init__.py:32, in choose_simulator(name, mixer_type, **kwargs)

repo_root = r"C:\new_fpga\updated_project\QAOAHP"
sys.path.insert(0, repo_root)
#sys.path.insert(0, r'C:\Users\N1259534\OneDrive - Nottingham Trent University\Desktop\Projects\QAOA_simulation_LV\QAOA_mtrix_simulation')
sys.executable


'c:\\Users\\N1259534\\AppData\\Local\\anaconda3\\envs\\QAOA_matrix\\python.exe'

### Test QAOA core component

In [ ]:
# Simple 2-node graph
G_simple = nx.Graph()
G_simple.add_edge((0, 1), (2,4), weight=1.0)

print(G_simple.edges(data=True))   


In [ ]:
# Simple 2-node graph
np.random.seed(10)
G_simple = nx.Graph()
for i in range(4):
    for j in range(i+1, 4):
        G_simple.add_edge(i, j, weight=1.0)

from main.Base.maxcut import get_maxcut_terms
# Get MaxCut terms
terms = get_maxcut_terms(G_simple)
print(f"Number of terms: {len(terms)}")
print("Terms (coefficient, qubits):")
for i , (coeff, qubits) in enumerate(terms):
    print(f" {i}: coeff= {coeff}, qubits={qubits}")

# Test with your current graph
print( f"\n\n graph G: {G_simple.number_of_nodes()} nodes, {G_simple.number_of_edges()} edge")

terms_g = get_maxcut_terms(G_simple)
print(f" generated {len(terms_g)} terms")
nx.draw(G_simple, with_labels=True)

In [ ]:
from main.Base import choose_simulator
import numpy as np

# Simple problem
N = 3

terms_simple = [(1.0, (0, 2)),(1, (1, 2))] 
#terms = [(np.random.normal(), spin_pair) for spin_pair in combinations(range(N), r=2)]


print(f"Terms is : {terms_simple}")
print("\n")

simmulator = choose_simulator('auto')

sim = simmulator(N, terms = terms_simple)

#get cost diogonal 
cost_diag = sim.get_cost_diagonal()
print(f" The cost diogonal is : {cost_diag}")
print("\n")

for i, cost in enumerate(cost_diag):
    binary = format(i, f'0{N}b')
    print(f"  |{binary}⟩: {cost:8.4f}")

assert len(cost_diag) == 2**N, f" cost diagonal should have {2**N} elements"
print(f"\n✓ Cost diagonal has correct size: {len(cost_diag)}")

In [ ]:
# TEST 2.2: Test with your actual graph G
from main.QAOA_objective import get_qaoa_objective
from main.Base import choose_simulator
import numpy as np

obj = get_qaoa_objective(G = G_simple, simulator= "python")



In [ ]:
# Test single evaluation (p=2)
theta0 = np.array([0.1, 0.2, 0.3, 0.4])
print(f"\nTest theta: {theta0}")

cost_value = obj(theta0)
print(f"Cost Value: {cost_value:.8f}")

# Test multiple evaluations (for optimization)
print("\nTesting 5 random parameter sets:")
for i in range(5):
    theta_rand = np.random.rand(4)
    cost = obj(theta_rand)
    print(f"  Iteration {i+1}: theta={theta_rand}, cost={cost:.6f}")

print("\n✓ Objective function stable over multiple calls")

In [ ]:
# TEST 2.3: Parameter handling
from main import parameter_utils

# Test theta parameterization (default)
theta = np.array([0.1, 0.2, 0.3, 0.4])  # p=2
print(f" te theta is {theta}")

gamma, beta = parameter_utils.convert_to_gamma_beta(theta, parameterization='theta')

print(f"Converted_gamma = {gamma}")
print(f"Converted_beta = {beta}")

theta_reconstructed = np.hstack([gamma, beta])
print(f"Reconstructed theta: {theta_reconstructed}")

if np.allclose(theta, theta_reconstructed):
    print("\n✓ Parameter conversion is consistent")
else:
    print(" is not same")


### Test Integeration 

In [ ]:
import numba.cuda

# Simple graph
G_test = nx.Graph()
G_test.add_edge(0, 1, weight=1.0)
G_test.add_edge(1, 2, weight=1.0)

theta = np.array([0.5, 0.3]) # p =1

# create opjective function
obj = get_qaoa_objective(G = G_test, simulator= "auto")
#single cost value
cost_value = obj(theta)
print(f"objective value of cpu result is : {cost_value:.10f}")

# gpu (if available)
if numba.cuda.is_available():
    obj_gpu = get_qaoa_objective(G = G_test, simulator = "gpu")
    cost_value_gpu = obj_gpu(theta)
    print(f"gpu result objective value: {cost_value:.10f}")

    if np.isclose(cost_value, cost_value_gpu):
        print("\n✓ GPU and CPU results are consistent")
    else:
        print("\n✗ GPU and CPU results differ")
else:
    print("\n GPU is not available ")


In [ ]:
# test optimisation loop 
from scipy.optimize import minimize
from main.QAOA_objective import get_qaoa_objective
P = 2
theta_new = np.random.rand(2*P)
print(f" new theta is {theta_new}")
print(f"the initial cost value is {obj(theta_new):.6f}")

#iteration
iteration_count = [0]
def callback(xk):
    iteration_count[0] +=1
    if iteration_count[0] % 5 ==0:
        print(f"  Iteration {iteration_count[0]}: cost={obj(xk):.6f}")

# optimise
result = minimize(obj, theta_new, method= 'COBYLA', callback= callback,
                   options = {'maxiter': 50})

print(f"final theta {result.x}")
print(f"final cost value is {result.fun:.6f}")
print(f"success: {result.success}")
print(f"totall iteration: {iteration_count[0]}")

if result.fun < obj(theta_new):
    print("\n✓ Optimization improved the cost value")
else:
    print("\n✗ Optimization did not improve the cost value")


### Data validation  (FPGA ready)



In [2]:
from main.Base.Simulators.FPGA import Fpga_sim
from main.Base.maxcut import get_maxcut_terms
from main.Base import choose_simulator


N = 6
#G_fpga = nx.Graph()
#for i in range(N):
#        for j in range(i+1, N):
#            G_fpga.add_edge(i, j, weight=1.0)
G_fpga = nx.random_regular_graph(d=3, n=N, seed=10)

# Optional visualization
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(G_fpga)
#nx.draw(G_fpga, pos, with_labels=True, node_color="lightblue")
#plt.show()

<Figure size 500x500 with 0 Axes>

In [22]:
from main.QAOA_objective import get_qaoa_objective
terms = get_maxcut_terms(G_fpga)
fpga_config = { "port": "COM3", "baudrate": 115200, "max_qubits": 14}
a = get_qaoa_objective(N, G_fpga, simulator='FPGA', fpga_config=fpga_config) 
theta = np.random.rand(8)
theta
#cost_value = a(theta)


array([0.98471557, 0.7181617 , 0.25320044, 0.8796992 , 0.81186064,
       0.36704921, 0.428109  , 0.37677096])

In [24]:
from main import parameter_utils
#theta = [0.1, 0.2, 0.3, 0.4,0.42, 0.5, 0.6, 0.7]
gamma, beta = parameter_utils.convert_to_gamma_beta(theta, parameterization='theta')
cosb , sinb = parameter_utils.generate_mixer_sincos_fpga( beta, p=4)
cosb, sinb


(array([0.72556883, 0.35886276, 0.4151512 , 0.36791985]),
 array([0.6881496 , 0.93339034, 0.90975243, 0.92985751]))

In [12]:
for address, value in ((gamma, beta), 
                     (gamma, beta)):
            print(f"Address: {address}, Value: {value}")

Address: [0.1, 0.2, 0.3, 0.4], Value: [0.42, 0.5, 0.6, 0.7]
Address: [0.1, 0.2, 0.3, 0.4], Value: [0.42, 0.5, 0.6, 0.7]


In [2]:
P = 64
N = 61

SCALE = 1 << N
MIN_VAL = -(1 << (P - 1))
MAX_VAL = (1 << (P - 1)) - 1
MASK_64 = (1 << P) - 1


def float_to_fixed(a: float) -> int:
    """
    Convert float to signed Q3.61 fixed-point integer.
    Saturates if value is outside representable range.
    """
    scaled = int(round(float(a) * SCALE))

    if scaled < MIN_VAL:
        return MIN_VAL
    if scaled > MAX_VAL:
        return MAX_VAL

    return scaled

In [ ]:
# get terms and simulator 
terms = get_maxcut_terms(G_fpga)
print(f"term is {terms}" )

sim = choose_simulator(name='python')(n_qubits=N, terms=terms)
cost_diagonal = sim.get_cost_diagonal()

for i, cost in enumerate(cost_diagonal):
    print(f"  [{i}]: {cost:.10f}  (type: {type(cost).__name__})")

# Check data types
assert all(isinstance(c, (float, np.floating)) for c in cost_diagonal), \
    "All costs should be floats"
print("✓ All costs are float type")


# Check size
assert len(cost_diagonal) == 2**N, f"Should have {2**N} costs"
print(f"✓ Correct size: {len(cost_diagonal)}")

# Initial state
initial_state = np.ones(2**N, dtype=np.complex128) / np.sqrt(2**N)
print(f"\nInitial state:")
for i, amp in enumerate(initial_state):
    print(f"  [{i}]: {amp.real:.6f} + {amp.imag:.6f}j")


# Parameters
gamma = np.array([0.5])
beta = np.array([0.3])
print(f"\nParameters:")
print(f"  gamma: {gamma} (type: {gamma.dtype})")
print(f"  beta:  {beta} (type: {beta.dtype})")
print(f"  cos(β): {np.cos(beta[0]):.10f}")
print(f"  sin(β): {np.sin(beta[0]):.10f}")


### Test FPGA part

In [5]:
from main.Base.Simulators.FPGA.Fpga_sim import FpgaDriver

import numpy as np

# --------------------------------------------------------------------------
# Opcode table 
# --------------------------------------------------------------------------
OP_SEND1T, OP_SEND8T, OP_MOV_T2A = 1, 2, 3
OP_MOV_S2U = 8
OP_FETCH1U, OP_FETCH8U = 60, 61
OP_INC_A = 84
OP_WRITE_T2RAM, OP_READ_RAM2U = 111, 112
OP_SEND_CMD, OP_WRITE_T2_AG = 118, 119
qa_WAIT, qa_RUN = 1, 2

BRAM_PARAMS     = 0x0800_0000_0000_0000
BRAM_STATE_REAL = 0x1000_0000_0000_0000
BRAM_STATE_IMAG = 0x2000_0000_0000_0000
BRAM_COST_FUNC  = 0x0400_0000_0000_0000

_OPNAMES = {
    1: "SEND1T", 2: "SEND8T", 3: "MOV_T2A", 4: "MOV_T2B", 5: "MOV_A2U",
    6: "MOV_A2B", 7: "MOV_Info2U", 8: "MOV_S2U", 9: "MOV_T2P",
    60: "FETCH1U", 61: "FETCH8U", 80: "ADD_B2A", 81: "MUL_B2A",
    84: "INC_A", 111: "WRITE_T2RAM", 112: "READ_RAM2U",
    118: "SEND_CMD", 119: "WRITE_T2_AG",
}

# --------------------------------------------------------------------------
# Fake serial port: records TX, supplies canned RX
# --------------------------------------------------------------------------
class MockSerial:
    def __init__(self):
        self.tx = bytearray()

    def write(self, b):
        self.tx.extend(bytes(b))

    def read(self, n):
        # 1-byte read == status poll -> return qa_WAIT so the loop breaks once.
        # 8-byte read == result fetch -> return zeros (content irrelevant to TX).
        return bytes([qa_WAIT]) if n == 1 else bytes(n)

    def reset_input_buffer(self):  pass
    def reset_output_buffer(self): pass
    def close(self):               pass


# --------------------------------------------------------------------------
# Encoding helpers (match all_test.py exactly)
# --------------------------------------------------------------------------
_FIX_N = 61
_FIX_P = 64


def _op(x):
    return bytes([x])

def _u64(v):
    # addresses + AG register values: positive, < 2**63 -> identical under
    # signed or unsigned, and identical to driver's masked unsigned encoding.
    return (v & ((1 << 64) - 1)).to_bytes(8, "little")

def _fx(x):
    # Q3.61 fixed-point, saturating (all_test.py float_to_fixed) -- identical to
    # the driver's _send_fixed for in-range values.
    scaled = int(round(x * (1 << _FIX_N)))
    lo, hi = -(1 << (_FIX_P - 1)), (1 << (_FIX_P - 1)) - 1
    scaled = max(lo, min(hi, scaled))
    return scaled.to_bytes(8, "little", signed=True)

def _mask64(a):
    return (1 << 64) - 1 if a >= 64 else (1 << a) - 1

# --------------------------------------------------------------------------
# Timing model (independent copy of all_test.py lines 79-136)
# --------------------------------------------------------------------------
def _timing(NQ, Np):
    LP_BRAM_A, LP_BRAM_D, LP_GEN_COST = 2, 1, 2
    LP_MIXER_IN, LP_MIXER_OUT = 1, 1
    L_BRAM_R = L_BRAM_W = LP_BRAM_A + LP_BRAM_D + 2
    N3 = 1 + 10 + 2 + 1 + 1 + 1
    gcN0, gcN1 = 10, 170
    gcPipe = 1 + gcN0 + 1 + gcN1 + 1
    Lc = 1 + gcPipe + 1 + L_BRAM_R + 2 * LP_GEN_COST
    Lm = 1 + N3 + 1 + L_BRAM_R + L_BRAM_W + 1 + LP_MIXER_IN + LP_MIXER_OUT
    LInit = 24
    NS = 1 << NQ
    LPipe = NS
    tl = Lm + NS // 2 + NS % 2
    if tl >= NS:
        LPipe = tl
    DVTc = Lc // LPipe + 1
    tGenCost = DVTc * LPipe - Lc
    if tGenCost < LInit:
        tGenCost += LPipe
    tbGenCost = LPipe * (NQ + 1) - Lc
    t_Mixer = LPipe
    if tbGenCost < LInit:
        t_Mixer = LPipe + LInit - tbGenCost
        tbGenCost = LInit
    t_Compute = t_Mixer + LPipe * NQ + Lm
    return dict(
        t_L2Addr=NS - 2, t_L2Pipe=LPipe - 2, t_L2PipeGC=Lc - 2,
        tb_B2GenCost=tbGenCost - 2, t_B2GenCost=tGenCost - 2,
        nPLayer=Np, L1Qbit=NQ - 1, AddrMask=_mask64(NQ - 1),
        tb_B2Mixer=t_Mixer - 2, t_L2Compute=t_Compute,
    )

# --------------------------------------------------------------------------
# Reference TX stream (faithful to all_test.py data_array + transmit loop)
# --------------------------------------------------------------------------
def build_reference(NQ, Np, H, sv0_re, sv0_im, betas, gammas):
    NS = 1 << NQ
    t = _timing(NQ, Np)
    s = bytearray()

    # 1) park in WAIT
    s += _op(OP_SEND1T) + _op(qa_WAIT) + _op(OP_SEND_CMD)

    # 2) addr_gen config -- all_test.py order (selector, value)
    ag = [
        (0, t["t_L2Addr"]), (3, t["t_L2Pipe"]), (1, t["t_L2PipeGC"]),
        (2, t["tb_B2GenCost"]), (7, t["t_B2GenCost"]), (4, t["nPLayer"]),
        (5, t["L1Qbit"]), (6, t["AddrMask"]), (8, t["tb_B2Mixer"]),
        (9, t["t_L2Compute"]),
    ]
    for sel, val in ag:
        s += _op(OP_SEND1T) + _op(sel) + _op(OP_MOV_T2A)
        s += _op(OP_SEND8T) + _u64(val) + _op(OP_WRITE_T2_AG)

    # 3) params: p+1 triples, redundant first cos/sin, trailing gamma=-1
    cosb = np.cos(np.asarray(betas, float))
    sinb = np.sin(np.asarray(betas, float))
    cosb_w = [float(cosb[0])] + [float(c) for c in cosb]
    sinb_w = [float(sinb[0])] + [float(c) for c in sinb]
    gam_w = [float(g) for g in gammas] + [-1.0]
    s += _op(OP_SEND8T) + _u64(BRAM_PARAMS) + _op(OP_MOV_T2A)
    for k in range(Np + 1):
        for v in (cosb_w[k], sinb_w[k], gam_w[k]):
            s += _op(OP_SEND8T) + _fx(v) + _op(OP_WRITE_T2RAM) + _op(OP_INC_A)

    # 4/5/6) state real, state imag, cost
    for bank, vals in ((BRAM_STATE_REAL, sv0_re),
                       (BRAM_STATE_IMAG, sv0_im),
                       (BRAM_COST_FUNC, H)):
        s += _op(OP_SEND8T) + _u64(bank) + _op(OP_MOV_T2A)
        for i in range(NS):
            s += _op(OP_SEND8T) + _fx(float(vals[i])) + _op(OP_WRITE_T2RAM) + _op(OP_INC_A)

    # 7) run
    s += _op(OP_SEND1T) + _op(qa_RUN) + _op(OP_SEND_CMD)
    # 8) one status poll (HOST_WAIT, breaks on first qa_WAIT)
    s += _op(OP_MOV_S2U) + _op(OP_FETCH1U)
    # 9) back to WAIT
    s += _op(OP_SEND1T) + _op(qa_WAIT) + _op(OP_SEND_CMD)

    # 10/11) read back real, imag
    for bank in (BRAM_STATE_REAL, BRAM_STATE_IMAG):
        s += _op(OP_SEND8T) + _u64(bank) + _op(OP_MOV_T2A)
        for i in range(NS):
            s += _op(OP_READ_RAM2U) + _op(OP_FETCH8U) + _op(OP_INC_A)

    return bytes(s)





In [6]:
# --------------------------------------------------------------------------
# Decoder for human-readable diffs
# --------------------------------------------------------------------------
def decode(stream):
    """Return list of (offset, text). Tracks SEND8T/SEND1T payloads."""
    out, i, n = [], 0, len(stream)
    while i < n:
        op = stream[i]
        name = _OPNAMES.get(op, f"?{op}")
        if op == OP_SEND8T and i + 9 <= n:
            payload = stream[i + 1:i + 9]
            sv = int.from_bytes(payload, "little", signed=True)
            out.append((i, f"SEND8T {payload.hex()}  (int={sv}, fx={sv / (1 << _FIX_N):+.6g})"))
            i += 9
        elif op == OP_SEND1T and i + 2 <= n:
            out.append((i, f"SEND1T 0x{stream[i + 1]:02x} ({stream[i + 1]})"))
            i += 2
        else:
            out.append((i, name))
            i += 1
    return out

def _report_mismatch(actual, reference):
    # first differing byte
    m = min(len(actual), len(reference))
    first = next((k for k in range(m) if actual[k] != reference[k]), m)
    print(f"  lengths: driver={len(actual)}  reference={len(reference)}")
    print(f"  first differing byte at offset {first}")
    da, dr = decode(actual), decode(reference)

    def window(dec):
        idx = next((j for j, (off, _) in enumerate(dec) if off >= first), len(dec) - 1)
        return dec[max(0, idx - 3): idx + 4]

    print("\n  --- DRIVER (around first diff) ---")
    for off, txt in window(da):
        print(f"    @{off:5d}  {txt}")
    print("\n  --- REFERENCE (around first diff) ---")
    for off, txt in window(dr):
        print(f"    @{off:5d}  {txt}")



# --------------------------------------------------------------------------
# Main entry point
# --------------------------------------------------------------------------
def run_parity_check(FpgaDriver, NQ=5, Np=2, seed=12345, verbose=True):
    rng = np.random.default_rng(seed)
    NS = 1 << NQ

    # Deterministic, in-range inputs (so neither side saturates/raises).
    H = rng.uniform(-1.0, 1.0, NS)
    sv0 = rng.normal(size=NS) + 1j * rng.normal(size=NS)
    sv0 /= np.linalg.norm(sv0)
    betas = rng.uniform(-np.pi / 4, np.pi / 4, Np)
    gammas = rng.uniform(-1.0, 1.0, Np)
    sv0_re = [sv0[i].real for i in range(NS)]
    sv0_im = [sv0[i].imag for i in range(NS)]
    cosb = np.cos(betas)
    sinb = np.sin(betas)

    # Build the driver and inject the fake port (bypass connect()).
    drv = FpgaDriver({"port": "mock", "baudrate": 115200, "timeout": 1})
    drv.ser = MockSerial()
    drv.connected = True

    drv.load_data(H, sv0_re, sv0_im, gammas, betas, cosb, sinb)
    drv.execute(Np)
    drv.read_result(NS)
    actual = bytes(drv.ser.tx)

    reference = build_reference(NQ, Np, H, sv0_re, sv0_im, betas, gammas)

    ok = actual == reference
    if verbose:
        print(f"[parity] NQ={NQ} Np={Np}  driver={len(actual)}B  reference={len(reference)}B")
        if ok:
            print("[parity] PASS - byte-for-byte identical to all_test.py")
        else:
            print("[parity] FAIL")
            _report_mismatch(actual, reference)
    return ok

In [9]:
run_parity_check(FpgaDriver, NQ=8, Np=2)

Loading data to FPGA...
  Phase 1: Reset cycle...
  Phase 2: Load parameters...
  Phase 3: Load state and cost data...
✓ Data loaded to FPGA
Reading 256 state amplitudes from FPGA...
✓ Read 256 amplitudes
[parity] NQ=8 Np=2  driver=10284B  reference=10284B
[parity] PASS - byte-for-byte identical to all_test.py


True